# Pipeline-Based Baseline — Google Colab

Implements the sequential pipeline (Sec 3.5.1) where:
- **Stage 1:** Train sarcasm detection model
- **Stage 2:** Feed sarcasm predictions into harmful intent classifier
- **CER:** Compute Cascade Error Rate to measure error propagation

**Before running:**
1. Set runtime to GPU: `Runtime > Change runtime type > T4 GPU`
2. Upload your data to Google Drive at: `MyDrive/mtl-bert/data/`
   - `data/sarcasm/sarcasm.csv`
   - `data/cyberbullying/cyberbullying.csv`
   - `data/emotions/emotions.csv`
3. Run all cells in order.

Results are saved to Drive so they survive session disconnects.

In [ ]:
# Install dependencies
!pip install -q transformers scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Change this if your Drive folder is named differently ──
BASE_DIR    = "/content/drive/MyDrive/mtl-bert"
DATA_DIR    = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results", "pipeline-baseline")

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Base dir : {BASE_DIR}")
print(f"Data dir : {DATA_DIR}")
print(f"Results  : {RESULTS_DIR}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import random
import json
import csv
from typing import List, Tuple

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Dataset helpers (inlined from dataset.py) ──────────────────────────

def create_sample_datasets():
    datasets = {}

    sarc_path = os.path.join(DATA_DIR, "sarcasm", "sarcasm.csv")
    sarc_data = []
    with open(sarc_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 3:
                try:
                    label = int(row[1])
                    text  = row[2].strip()
                    if text:
                        sarc_data.append((text, label))
                except ValueError:
                    continue
    datasets["sarc"] = sarc_data

    cyber_path = os.path.join(DATA_DIR, "cyberbullying", "cyberbullying.csv")
    intent_data = []
    with open(cyber_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 3:
                try:
                    label = int(row[1])
                    text  = row[2].strip()
                    if text:
                        intent_data.append((text, label))
                except ValueError:
                    continue
    datasets["intent"] = intent_data

    emo_path = os.path.join(DATA_DIR, "emotions", "emotions.csv")
    emotion_data = []
    with open(emo_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 2:
                try:
                    label = int(row[0])
                    text  = row[1].strip()
                    if text:
                        emotion_data.append((text, label))
                except ValueError:
                    continue
    datasets["emotion"] = emotion_data

    return datasets


def compute_metrics(predictions, labels):
    return {
        "accuracy" : accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average="weighted", zero_division=0),
        "recall"   : recall_score(labels, predictions, average="weighted", zero_division=0),
        "f1"       : f1_score(labels, predictions, average="weighted", zero_division=0),
    }

In [ ]:
# ── Reproducibility ────────────────────────────────────────────────────

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ── Datasets ────────────────────────────────────────────────────────────

class SingleTaskDataset(Dataset):
    def __init__(self, data: List[Tuple], tokenizer, max_length=128):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.samples    = data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, label = self.samples[idx]
        encoding = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt',
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label'         : torch.tensor(label, dtype=torch.long),
        }


class PipelineHarmDataset(Dataset):
    """Harm dataset augmented with upstream sarcasm predictions (Sec 3.5.1)."""
    def __init__(self, data: List[Tuple], sarc_probs: List[float],
                 tokenizer, max_length=128):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.samples    = data
        self.sarc_probs = sarc_probs

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, label = self.samples[idx]
        encoding = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt',
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label'         : torch.tensor(label, dtype=torch.long),
            'sarc_prob'     : torch.tensor(self.sarc_probs[idx], dtype=torch.float),
        }


# ── Models ──────────────────────────────────────────────────────────────

class SingleTaskBERT(nn.Module):
    def __init__(self, model_name: str, num_classes: int):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.encoder.config.hidden_size
        self.num_classes = num_classes
        self.classifier = nn.Linear(hidden_size, 1 if num_classes == 2 else num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(outputs.last_hidden_state[:, 0])


class PipelineHarmBERT(nn.Module):
    """Pipeline harm classifier: z = h_[CLS] || p_sarc -> y_harm = sigma(Wz + b)"""
    def __init__(self, model_name: str):
        super().__init__()
        self.encoder   = AutoModel.from_pretrained(model_name)
        hidden_size    = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size + 1, 1)

    def forward(self, input_ids, attention_mask, sarc_prob):
        outputs  = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb  = outputs.last_hidden_state[:, 0]
        combined = torch.cat([cls_emb, sarc_prob.unsqueeze(-1)], dim=-1)
        return self.classifier(combined)


# ── Training helpers ────────────────────────────────────────────────────

def train_binary_model(model, train_loader, val_loader, dev, num_epochs=5,
                       is_pipeline=False):
    optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    loss_fn   = nn.BCEWithLogitsLoss()
    best_val_f1 = 0.0
    best_state  = None

    for epoch in range(num_epochs):
        model.train()
        total_loss, n_batches = 0.0, 0

        for batch in train_loader:
            input_ids      = batch['input_ids'].to(dev)
            attention_mask = batch['attention_mask'].to(dev)
            labels         = batch['label'].to(dev).float()

            if is_pipeline:
                sarc_prob = batch['sarc_prob'].to(dev)
                logits = model(input_ids, attention_mask, sarc_prob).squeeze(-1)
            else:
                logits = model(input_ids, attention_mask).squeeze(-1)

            loss = loss_fn(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

        val_metrics, _, _ = evaluate_binary(model, val_loader, dev, is_pipeline)
        print(f"    Epoch {epoch+1}/{num_epochs} - "
              f"Loss: {total_loss/max(n_batches,1):.4f} - "
              f"Val Acc: {val_metrics['accuracy']:.4f}, "
              f"Val F1: {val_metrics['f1']:.4f}")

        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    return best_state, {'best_val_f1': best_val_f1}


def evaluate_binary(model, dataloader, dev, is_pipeline=False):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch['input_ids'].to(dev)
            attention_mask = batch['attention_mask'].to(dev)
            if is_pipeline:
                sarc_prob = batch['sarc_prob'].to(dev)
                logits = model(input_ids, attention_mask, sarc_prob).squeeze(-1)
            else:
                logits = model(input_ids, attention_mask).squeeze(-1)
            preds = (torch.sigmoid(logits) > 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['label'].numpy())
    return compute_metrics(all_preds, all_labels), all_preds, all_labels


def get_sarcasm_predictions(sarc_model, texts, tokenizer, dev,
                            max_length=128, batch_size=32):
    sarc_model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        encoding = tokenizer(
            batch_texts, truncation=True, padding='max_length',
            max_length=max_length, return_tensors='pt',
        )
        with torch.no_grad():
            logits = sarc_model(encoding['input_ids'].to(dev),
                                encoding['attention_mask'].to(dev)).squeeze(-1)
            probs = torch.sigmoid(logits)
        all_probs.extend(probs.cpu().numpy().tolist())
    return all_probs


def evaluate_pipeline_harm_with_override(model, test_data, sarc_probs_actual,
                                         tokenizer, dev, max_length=128,
                                         batch_size=16):
    ds_actual = PipelineHarmDataset(test_data, sarc_probs_actual, tokenizer, max_length)
    dl_actual = DataLoader(ds_actual, batch_size=batch_size, shuffle=False)
    _, preds_with, labels = evaluate_binary(model, dl_actual, dev, is_pipeline=True)

    sarc_zeros = [0.0] * len(test_data)
    ds_clean = PipelineHarmDataset(test_data, sarc_zeros, tokenizer, max_length)
    dl_clean = DataLoader(ds_clean, batch_size=batch_size, shuffle=False)
    _, preds_without, _ = evaluate_binary(model, dl_clean, dev, is_pipeline=True)

    return preds_with, preds_without, labels


def compute_cer(preds_with_sarc, preds_without_sarc, labels):
    total = len(labels)
    caused_errors = 0
    for i in range(total):
        if (preds_with_sarc[i] != labels[i]) and (preds_without_sarc[i] == labels[i]):
            caused_errors += 1
    return caused_errors / total if total > 0 else 0.0

In [ ]:
# ── Main training ──────────────────────────────────────────────────────

print("=" * 60)
print("  Pipeline-Based Baseline (Sec 3.5.1) - Multi-Seed Training")
print("=" * 60)

MODEL_NAME = "bert-base-uncased"
SEEDS      = [42, 123, 456]
BATCH_SIZE = 16
NUM_EPOCHS = 5
MAX_LENGTH = 128

print(f"\nLoading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nLoading datasets...")
tasks_data = create_sample_datasets()
for task_name, data in tasks_data.items():
    print(f"  {task_name}: {len(data)} samples")

# Resume support
progress_path = os.path.join(RESULTS_DIR, "training_progress.json")
all_results   = {"sarc": [], "intent": [], "cer": []}
completed     = set()

if os.path.exists(progress_path):
    with open(progress_path, "r") as f:
        progress = json.load(f)
    completed   = set(progress.get("completed", []))
    all_results = progress.get("results", all_results)
    if completed:
        print(f"\nResuming: {len(completed)} seed(s) already done.")

for seed_idx, seed in enumerate(SEEDS):
    seed_key = f"seed{seed}"
    if seed_key in completed:
        print(f"\n--- Skipping seed {seed} (already done) ---")
        continue

    print(f"\n{'='*60}")
    print(f"  Seed {seed_idx+1}/{len(SEEDS)} (seed={seed})")
    print(f"{'='*60}")
    set_seed(seed)

    # Split all datasets (80/10/10)
    train_data, val_data, test_data = {}, {}, {}
    for task_name, task_samples in tasks_data.items():
        shuffled = task_samples.copy()
        random.shuffle(shuffled)
        n = len(shuffled)
        s1, s2 = int(0.8 * n), int(0.9 * n)
        train_data[task_name] = shuffled[:s1]
        val_data[task_name]   = shuffled[s1:s2]
        test_data[task_name]  = shuffled[s2:]

    # ======== Stage 1: Train sarcasm model ========
    print(f"\n  [Stage 1] Training sarcasm model...")
    print(f"    Train: {len(train_data['sarc'])}, Val: {len(val_data['sarc'])}")

    sarc_model    = SingleTaskBERT(MODEL_NAME, 2).to(device)
    sarc_train_dl = DataLoader(SingleTaskDataset(train_data['sarc'], tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
    sarc_val_dl   = DataLoader(SingleTaskDataset(val_data['sarc'],   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
    sarc_test_dl  = DataLoader(SingleTaskDataset(test_data['sarc'],  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

    best_sarc_state, _ = train_binary_model(sarc_model, sarc_train_dl, sarc_val_dl, device, num_epochs=NUM_EPOCHS)
    sarc_model.load_state_dict(best_sarc_state)
    sarc_model.to(device)

    sarc_test_metrics, _, _ = evaluate_binary(sarc_model, sarc_test_dl, device)
    print(f"    Sarcasm test: Acc={sarc_test_metrics['accuracy']:.4f}, F1={sarc_test_metrics['f1']:.4f}")

    # ======== Stage 2: Pipeline harm model ========
    print(f"\n  [Stage 2] Training pipeline harm model...")
    print(f"    Train: {len(train_data['intent'])}, Val: {len(val_data['intent'])}")

    harm_train_texts = [t for t, _ in train_data['intent']]
    harm_val_texts   = [t for t, _ in val_data['intent']]
    harm_test_texts  = [t for t, _ in test_data['intent']]

    print("    Getting sarcasm predictions on harm data...")
    sarc_probs_train = get_sarcasm_predictions(sarc_model, harm_train_texts, tokenizer, device, MAX_LENGTH)
    sarc_probs_val   = get_sarcasm_predictions(sarc_model, harm_val_texts,   tokenizer, device, MAX_LENGTH)
    sarc_probs_test  = get_sarcasm_predictions(sarc_model, harm_test_texts,  tokenizer, device, MAX_LENGTH)

    sarc_pos_rate = np.mean([1 if p > 0.5 else 0 for p in sarc_probs_test])
    print(f"    Sarcasm positive rate on harm test: {sarc_pos_rate:.2%}")

    harm_train_dl = DataLoader(PipelineHarmDataset(train_data['intent'], sarc_probs_train, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
    harm_val_dl   = DataLoader(PipelineHarmDataset(val_data['intent'],   sarc_probs_val,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

    harm_model = PipelineHarmBERT(MODEL_NAME).to(device)
    best_harm_state, _ = train_binary_model(harm_model, harm_train_dl, harm_val_dl, device, num_epochs=NUM_EPOCHS, is_pipeline=True)
    harm_model.load_state_dict(best_harm_state)
    harm_model.to(device)

    # ======== CER Analysis ========
    print(f"\n  [CER Analysis] Computing Cascade Error Rate...")

    preds_with, preds_without, harm_labels = evaluate_pipeline_harm_with_override(
        harm_model, test_data['intent'], sarc_probs_test, tokenizer, device, MAX_LENGTH, BATCH_SIZE,
    )

    harm_metrics       = compute_metrics(preds_with, harm_labels)
    harm_metrics_clean = compute_metrics(preds_without, harm_labels)
    cer = compute_cer(preds_with, preds_without, harm_labels)

    print(f"    Harm test (pipeline): Acc={harm_metrics['accuracy']:.4f}, F1={harm_metrics['f1']:.4f}")
    print(f"    Harm test (no sarc):  Acc={harm_metrics_clean['accuracy']:.4f}, F1={harm_metrics_clean['f1']:.4f}")
    print(f"    CER = {cer:.4f} ({cer:.2%} of predictions corrupted by sarcasm)")

    total = len(harm_labels)
    print(f"    Errors with sarcasm: {sum(1 for i in range(total) if preds_with[i] != harm_labels[i])}/{total}")
    print(f"    Errors without sarcasm: {sum(1 for i in range(total) if preds_without[i] != harm_labels[i])}/{total}")

    # Save seed results
    all_results['sarc'].append(sarc_test_metrics)
    all_results['intent'].append(harm_metrics)
    all_results['cer'].append({'cer': cer,
                                'harm_f1_with_sarc': harm_metrics['f1'],
                                'harm_f1_without_sarc': harm_metrics_clean['f1']})
    completed.add(seed_key)

    with open(progress_path, 'w') as f:
        json.dump({'completed': list(completed), 'results': all_results}, f, indent=2)
    print(f"  Progress saved ({len(completed)}/{len(SEEDS)} seeds done)")

In [ ]:
# ── Aggregated results ─────────────────────────────────────────────────

print(f"\n{'='*60}")
print(f"  Pipeline Baseline: Aggregated Results ({len(SEEDS)} seeds)")
print(f"{'='*60}")

aggregated = {}
for task in ['sarc', 'intent']:
    agg = {}
    for metric in ['accuracy', 'precision', 'recall', 'f1']:
        values = [m[metric] for m in all_results[task]]
        agg[metric] = {'mean': float(np.mean(values)), 'std': float(np.std(values))}
    aggregated[task] = agg
    print(f"\n  {task}:")
    for metric in ['accuracy', 'precision', 'recall', 'f1']:
        print(f"    {metric:>10s}: {agg[metric]['mean']:.4f} +/- {agg[metric]['std']:.4f}")

# CER summary
cer_values = [c['cer'] for c in all_results['cer']]
aggregated['cer'] = {
    'mean': float(np.mean(cer_values)),
    'std': float(np.std(cer_values)),
    'per_seed': all_results['cer'],
}
print(f"\n  Cascade Error Rate (CER):")
print(f"    CER: {np.mean(cer_values):.4f} +/- {np.std(cer_values):.4f}")
print(f"    (MTL CER = 0 by design, no sequential dependency)")

aggregated['emotion_note'] = (
    "Emotion is trained independently in the pipeline (same as single-task "
    "baseline). Refer to STL-BERT results for emotion metrics."
)
print(f"\n  Note: Emotion task is identical to STL baseline in pipeline setup.")

with open(os.path.join(RESULTS_DIR, 'aggregated_results.json'), 'w') as f:
    json.dump(aggregated, f, indent=2)

print(f"\n  Done!")